# 实验 07：CIFAR-10 卷积 VAE
目标：重参数化、重构项与 KL 项、验证模型选择、测试和先验采样。
参考：[VAE 原始论文](https://arxiv.org/abs/1312.6114)、[PyTorch VAE 示例](https://github.com/pytorch/examples/tree/main/vae)。
本实现将 VAE 改为处理 CIFAR-10 的 32×32 RGB 卷积结构，不切换第三个数据集。


In [ ]:
from pathlib import Path
import os, sys

# 从项目根目录、notebooks/ 或它们的子目录启动时都能找到 labkit。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "labkit").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请将整个课程目录解压后，从项目根目录启动 JupyterLab")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)


## 1. 数据加载
训练/验证/测试划分与其他 CIFAR-10 实验相同，标签在生成模型训练中不使用。输入统一归一化至 [-1,1]。


In [ ]:
from datetime import datetime
import inspect
import torch
from IPython.display import display, Image as DisplayImage
from labkit.common import choose_device, seed_everything, save_json
from labkit.data import load_image_data
from labkit.generative_models import ConvVAE, DCGenerator, DCDiscriminator, dcgan_init, vae_loss
from labkit.generative import (fit_vae, fit_gan, vae_epoch, gan_train_step, gan_diagnostics,
                               save_vae_examples, save_grid, noise, plot_generative_history)
SMOKE = os.environ.get("LAB_SMOKE", "0") == "1"
SEED, LATENT_DIM, CHANNELS, BATCH_SIZE = 42, 64, 32, 128
device = choose_device("auto")
seed_everything(SEED)
limits = [128, 64, 64] if SMOKE else [0, 0, 0]
# 与分类实验共享相同的 CIFAR-10 划分，但生成任务不做随机增广。
bundle = load_image_data(ROOT / "data", BATCH_SIZE, SEED, augment=False,
                         max_train=limits[0], max_val=limits[1], max_test=limits[2])
print("device:", device, "数据规模:", bundle.meta["sizes"], "SMOKE:", SMOKE)


## 2. 模型与损失
编码器输出 `mu`、`logvar`。重参数化使用 `z = mu + exp(0.5*logvar)*eps`。
解码器以 Tanh 输出条件均值，使用固定标准差 0.1 的高斯像素似然。
重构负对数似然在每张图的全部像素上求和，KL 在潜变量维度求和，然后都对 batch 求均值。
`beta=1` 时为负 ELBO 的一次 Monte Carlo 估计。`beta!=1` 应称为 beta-VAE 目标。


In [ ]:
EPOCHS = 1 if SMOKE else 30
BETA, OBSERVATION_STD, LR = 1.0, 0.1, 0.001
RUN_DIR = ROOT / "runs" / datetime.now().strftime("vae_%Y%m%d_%H%M%S_%f")
config = {"task": "vae", "model": "conv_vae", "epochs": EPOCHS, "lr": LR,
          "latent_dim": LATENT_DIM, "channels": CHANNELS, "seed": SEED,
          "beta": BETA, "observation_std": OBSERVATION_STD, "smoke": SMOKE,
          "batch_size": BATCH_SIZE, "torch_version": str(torch.__version__)}
model = ConvVAE(LATENT_DIM, CHANNELS).to(device)
print(model)
print(inspect.getsource(vae_loss))
print(inspect.getsource(ConvVAE.reparameterize))


## 3. 完整训练与验证
按验证目标选择最佳 epoch，保存到 best.pt。验证只访问官方训练集划分出的 5,000 张图，不访问测试集。


In [ ]:
print(inspect.getsource(vae_epoch))
print(inspect.getsource(fit_vae))


In [ ]:
history = fit_vae(model, bundle, device, RUN_DIR, config)
plot_generative_history(history, "vae", RUN_DIR / "learning_curves.png")


## 4. 最终测试与重构
测试负 ELBO 使用固定种子的一次后验采样估计。重构 MSE / PSNR 使用 `decode(mu)`，在 [0,1] 显示空间计算。
PSNR 为 `-10*log10(全部像素平均 MSE)`，它不能衡量先验随机生成的质量。
连续高斯密度可以大于 1，含常数项的 NLL 可能为负。


In [ ]:
metrics = vae_epoch(model, bundle.test, device, beta=BETA, observation_std=OBSERVATION_STD)
save_json(RUN_DIR / "test_metrics.json", metrics)
print(metrics)
save_vae_examples(model, bundle.test, device, RUN_DIR, "test", SEED + 2000)
# 上排原图，下排使用潜变量均值解码的重构图。
display(DisplayImage(filename=str(RUN_DIR / "test_reconstruction.png")))


## 5. 先验生成
从 N(0,I) 采样潜变量，显示解码器条件均值。生成阶段不需要编码器或原始输入图。


In [ ]:
display(DisplayImage(filename=str(RUN_DIR / "test_samples.png")))
print("独立进程生成命令:")
print(f'python generate.py --run-dir "{RUN_DIR}" --seed 2026')


## 6. 实操练习
1. 潜维度使用 16、64、128，比较重构和先验生成。
2. 改变 beta，分别画出 KL 和重构项，记录潜变量是否遭到忽略。
3. 使用两个训练图像的 mu 做线性插值并解码，观察中间变化。
4. 解释“重构好”为什么不保证“先验采样好”。每次变更使用新的运行目录。
